In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [2]:
trainIDs =  ["F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100epat2_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100epat2_seed1993_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100epat2_seed42_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100ep_seed1994_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",
            "F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100ep_seed2023_L1_4ChTrans128fold0_prec32_pythaemodel-custom_factor_vae",]

res_root = r"/project/ukbblatent/Out/Results"
gwas_folder = "GWAS_fullDSV2"

In [3]:
print("Reading the latents.....")
processed_latents = []
for trainID in trainIDs:
    df = pd.read_table(f"{res_root}/{trainID}/{gwas_folder}/processed_raw.tsv").sort_values("FID").reset_index(drop=True)
    IID = df.IID
    FID = df.FID
    df.drop(columns=["IID", "FID"], inplace=True)
    latents = df.columns.to_list()
    processed_latents.append(df)

print("Merging the latents with PCA...")
concatenated_latents = np.concatenate(processed_latents, axis=1)
pca = PCA(n_components=len(latents))
final_components = pca.fit_transform(concatenated_latents)

merged_latents = pd.DataFrame(final_components, columns=latents)
merged_latents['IID'] = IID
merged_latents['FID'] = FID

cols = ['FID', 'IID'] + [col for col in merged_latents.columns if col not in ['FID', 'IID']]
merged_latents = merged_latents[cols]

Reading the latents.....
Merging the latents with PCA...


In [4]:
merged_latents.to_csv("/group/glastonbury/soumick/GWAS/merged_models/5Seeds_FVAE_128_Crop3D_DSV2_fold0/pca/merged_latents_raw.tsv", sep="\t", index=False)